In [ ]:
# Run first on Google Colab
!pip install qutip -q

# §6 Hamiltonian Simulation

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Apply Stone's theorem to identify generators of quantum gates
- Implement first-order Trotter and second-order Strang splitting
- Plot Trotter error vs step count and verify O(1/r) and O(1/r²) convergence
- Simulate a sparse Hamiltonian and verify the 1/r error scaling


In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from scipy.linalg import expm

print(f"QuTiP {qt.__version__}")

# ── Helpers from Notebook 1 ───────────────────────────────────────────────────
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n; ops[qubit] = gate
    return qt.tensor(ops)

def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0)*qt.basis(2,0).dag()
    P1 = qt.basis(2,1)*qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)]*n; ops0[ctrl] = P0
    ops1 = [qt.qeye(2)]*n; ops1[ctrl] = P1; ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

def swap_gate(n, i, j):
    N = 2**n; mat = np.zeros((N,N),dtype=complex)
    for k in range(N):
        bits = list(format(k,f'0{n}b')); bits[i],bits[j]=bits[j],bits[i]
        mat[int(''.join(bits),2),k]=1.0
    return qt.Qobj(mat, dims=[[2]*n,[2]*n])

def qft_circuit(n):
    U = qt.tensor([qt.qeye(2)]*n); H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1,n,i)*U
        for j in range(i+1,n):
            m=j-i+1; Rm=qt.Qobj(np.diag([1.0,np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm,n,j,i)*U
    for i in range(n//2): U = swap_gate(n,i,n-1-i)*U
    return U

def grover_operators(n, marked):
    N = 2**n
    dims = [[2]*n, [2]*n]
    diag = np.ones(N)
    for m in marked:
        diag[m] = -1.0
    O_f = qt.Qobj(np.diag(diag), dims=dims)
    zero_n = qt.tensor([qt.basis(2,0)]*n)
    s = qt.gates.hadamard_transform(n) * zero_n
    D = 2 * s * s.dag() - qt.tensor([qt.qeye(2)]*n)
    return O_f, D, s

def run_grover(n, marked, n_steps=None):
    N = 2**n; M = len(marked)
    theta = np.arcsin(np.sqrt(M/N))
    k_opt = int(np.floor(np.pi / (4*theta)))
    if n_steps is None:
        n_steps = k_opt
    O_f, D, s = grover_operators(n, marked)
    G = D * O_f
    psi = s.copy(); states = [psi]
    for _ in range(n_steps):
        psi = G * psi; states.append(psi.copy())
    return states, k_opt, theta

print("Grover helpers defined.")


In [ ]:
#@title 🔒 Solution vault — enter the key from class to unlock  {display-mode: "form"}
# Solutions are obfuscated. Use the unlock key your instructor hands out in
# class, then run the relevant exercise's cell, e.g. reveal_solution("1.1", key="...").
import base64, zlib

_VAULT = {
    "6.1": "PS16HBvtRm0pV7VSJvCIIzBCA3PL9r4k0UfZm1QprqoUALOX+57jGDU2mm9FIqFTQCJvELE7rpGU0XPj9YYSMCmDoJ/LLviE9RrsBFztH1IcMUbVQ7z53TD5rb8BFdvWWa8JNxUzOp0pRG4vKyRABvzWs/6XYxZX00ym6eS4b9w7NKkoLO52R5ad4JDCJtzgh1UCIOxZbgYuoUOzYgmWGu/Fv8L2HSS/FieA3Kkiit8xplLRPhJR+5KbwHDOQ8S1SZ4YvHdDq/U6K+d7IlDvQ/lLDhzL8UF7ynZCCOvT7aOMFlZpdgGKYKBUKuPXcjBRBONUkow2zWPyOPbr+Doi91MyajL6bfe3N6SF39dr4HGBGcGK/4dVIdjEhVZsCI3PQvbekt1sOBa7mo8hFuP76ItK/IvRlPKYBsu7RQLciCuw3wkV4e4PeNgB/H/mlIKNvDFA7C4A/FErpx5YFv6p7pTbbaYJexkeeiwfscJCVhv/ZkDwLc+wumIGV69WZRnIeq9YAPvz2kSEoUr/oejEW7Jc5j9wFH5r2m+nMBXTWvVvyPgqiD0yAwIqDVbsZg50FMMMC5/M0h8jE2UEAv4rF9ZtNj4VshILqlpBCZ9SFjYBEMe9DOq2PMH4QOpgQ5i7me6oFMw441T475iuLZ6cVlDBZ9UR3yCWlgnyj8TupL/3PhS8U2RTPZl0juXQcoN7HZ9vO0TMV3fp+KUY4OCzFlt1GMsTdanO6zE7WpnOweSi6PwAu6Yn2PASt8oyix+hZxFfComUeGMzTa0RvLV0ugjN1UilQbQ8wQdjia49Z78ltq6lRZE1FFq0uiXcPl8+KaLRbgLEW7KwukMEt2cQcX/T8FI/ZDWGjcbQYDTSYjMeVwg=",
}

def reveal_solution(exercise, key=""):
    """Print the solution for `exercise` (e.g. "1.1"), given the class unlock key."""
    blob = _VAULT.get(str(exercise))
    if blob is None:
        print(f"No solution stored for exercise {exercise!r}."); return
    kb = key.strip().encode()
    if not kb:
        print("Enter the key your instructor gave you, e.g. "
              f'reveal_solution("{exercise}", key="...").'); return
    data = base64.b64decode(blob.encode())
    out = bytes(b ^ kb[i % len(kb)] for i, b in enumerate(data))
    if out[:4] != b"SOL1":
        print("✗ Wrong key — check the key your instructor gave you for THIS notebook."); return
    print(zlib.decompress(out[4:]).decode("utf-8"))


---
## Part 2: Hamiltonian Simulation

### 6.1 Stone's theorem and unitary generators

**Theorem (Stone):** Every strongly continuous one-parameter unitary group $U_t = e^{iHt}$ corresponds to a unique Hermitian generator $H = H^\dagger$.

Every quantum gate $U$ is of this form. The eigendecomposition $H = V\Lambda V^\dagger$ gives
$e^{iHt} = V\,\mathrm{diag}(e^{i\lambda_1 t},\ldots)\,V^\dagger$.

**Grover operator revisited:** In the $\{|\alpha\rangle,|\beta\rangle\}$ plane, $G$ acts as rotation by $2\theta$, so
$$G = e^{-2i\theta\,\sigma_y} \implies \mathcal{G}|_{V_{\alpha\beta}} = -2\theta\,\sigma_y$$
where $\sigma_y$ is Pauli-$Y$ in the $\{|\alpha\rangle,|\beta\rangle\}$ basis.


In [ ]:
# ── Verify: G = exp(-2i*theta*sigma_y) in the Grover subspace ─────────────────
n_st = 3;  marked_st = [5]
N_st = 2**n_st
theta_st = np.arcsin(np.sqrt(1/N_st))

O_f_st, D_st, s_st = grover_operators(n_st, marked_st)
G_st = D_st * O_f_st

# Compute G in {|alpha>, |beta>} basis (2x2)
alpha_st = qt.Qobj(np.array([1/np.sqrt(N_st-1) if x not in marked_st else 0.
                               for x in range(N_st)]), dims=[[2]*n_st,[1]*n_st])
beta_st  = qt.Qobj(np.array([1/np.sqrt(1) if x in marked_st else 0.
                               for x in range(N_st)]), dims=[[2]*n_st,[1]*n_st])

G_2x2 = np.array([[complex(v.dag()*G_st*u)
                    for u in [alpha_st, beta_st]]
                   for v in [alpha_st, beta_st]])

# Generator: G_2x2 = exp(-2i*theta * sigma_y)
# Stone generator: G = exp(i * generator), so generator = -2*theta*sigma_y
sigma_y_2x2 = np.array([[0, -1j],[1j, 0]])
G_from_gen = expm(-2j * theta_st * sigma_y_2x2)

print("G in {|α⟩,|β⟩} basis:")
print(np.round(G_2x2, 4))
print("\nexp(-2iθ σ_y) =")
print(np.round(G_from_gen, 4))
print("\nMatch:", np.allclose(G_2x2, G_from_gen, atol=1e-6))
print(f"\nθ = {np.degrees(theta_st):.3f}°,  rotation angle 2θ = {np.degrees(2*theta_st):.3f}°")


### 6.2 Product formulas (Trotter splitting)

**Problem:** Simulate $e^{-iHt}$ where $H = A + B$ and $e^{-iAt}$, $e^{-iBt}$ are each easy to compute.

**First-order Trotter (Lie–Trotter):**
$$e^{-i(A+B)t} = \lim_{r\to\infty}\left(e^{-iAt/r}e^{-iBt/r}\right)^r$$

**Error:** $\|e^{-i(A+B)t} - (e^{-iAt/r}e^{-iBt/r})^r\| = O(\|[A,B]\|t^2/r)$

**Second-order Strang splitting:**
$$e^{-i(A+B)t} \approx \left(e^{-iAt/2r}e^{-iBt/r}e^{-iAt/2r}\right)^r$$

**Error:** $O(\|[[A,B],A]\|t^3/r^2)$ — one order better in $r$.


In [ ]:
# ── Set up a 2-sparse test Hamiltonian: H = A + B ─────────────────────────────
# Use a 4-qubit system; A and B are simple Pauli tensor products
n_ham = 2   # 2 qubits for clarity
I = qt.qeye(2);  X = qt.sigmax();  Z = qt.sigmaz()

# H = J*(Z⊗Z) + h*(X⊗I + I⊗X)  — transverse-field Ising model
# A and B do NOT commute, so Trotter has non-trivial error to plot
J, h = 1.0, 0.8
A = J * qt.tensor(Z, Z)                          # Ising ZZ coupling
B = h * (qt.tensor(X, I) + qt.tensor(I, X))     # transverse field
H = A + B

t_sim = 2.0   # total simulation time

# ── Exact evolution: matrix exponential ───────────────────────────────────────
def exact_evolution(H, t):
    return (-1j * H * t).expm()

U_exact = exact_evolution(H, t_sim)

# ── First-order Trotter (r steps) ─────────────────────────────────────────────
def trotter1(A, B, t, r):
    """First-order Trotter: (e^{-iAt/r} e^{-iBt/r})^r"""
    dt = t / r
    U_A = exact_evolution(A, dt)
    U_B = exact_evolution(B, dt)
    U_step = U_B * U_A      # right-to-left: apply A first, then B
    U = U_step
    for _ in range(r-1):
        U = U_step * U
    return U

# ── Second-order Strang splitting ─────────────────────────────────────────────
def trotter2(A, B, t, r):
    """Strang splitting: (e^{-iAt/2r} e^{-iBt/r} e^{-iAt/2r})^r"""
    dt = t / r
    U_Ah = exact_evolution(A, dt/2)
    U_B  = exact_evolution(B, dt)
    U_step = U_Ah * U_B * U_Ah
    U = U_step
    for _ in range(r-1):
        U = U_step * U
    return U

# ── Error vs step count r ─────────────────────────────────────────────────────
r_vals = [1, 2, 4, 8, 16, 32, 64]
err1 = []
err2 = []

for r in r_vals:
    U1 = trotter1(A, B, t_sim, r)
    U2 = trotter2(A, B, t_sim, r)
    err1.append((U_exact - U1).norm())
    err2.append((U_exact - U2).norm())

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(r_vals, err1, 'o-', label='1st order Trotter  (O(1/r))')
ax.loglog(r_vals, err2, 's-', label='2nd order Strang   (O(1/r²))')
# Reference lines
ax.loglog(r_vals, [err1[0]/r for r in r_vals], 'k--', alpha=0.4, label='∝ 1/r')
ax.loglog(r_vals, [err2[1]*r_vals[1]**2/r**2 for r in r_vals], 'k:', alpha=0.4, label='∝ 1/r²')
ax.set_xlabel('Trotter steps r');  ax.set_ylabel('operator norm error')
ax.set_title(f'Trotter error: H = J(Z⊗Z) + h(X⊗I+I⊗X), J={J}, h={h}, t={t_sim}')
ax.legend();  plt.tight_layout();  plt.show()
print(f"Commutator ‖[A,B]‖ = {(A*B - B*A).norm():.4f}")


### Exercise 6.1 — First-order Trotter error bound

**(a)** The first-order Trotter error satisfies
$$\|e^{-i(A+B)t} - (e^{-iAt/r}e^{-iBt/r})^r\| \leq \frac{t^2}{2r}\|[A,B]\|$$
for any $r \geq 1$. Verify this bound numerically for the Hamiltonian above.

**(b)** Consider the Heisenberg chain on $n=3$ sites with **open** boundary conditions,
$$H = H_1 + H_2, \quad H_1 = J(X_1X_2+Y_1Y_2+Z_1Z_2), \quad H_2 = J(X_2X_3+Y_2Y_3+Z_2Z_3),$$
i.e. just the two bonds $(1,2)$ and $(2,3)$ — there is no $(3,1)$ bond.
The natural Trotter split takes $A=H_1$, $B=H_2$. Compute the commutator $\|[H_1, H_2]\|$;
it is non-zero only because $H_1$ and $H_2$ share site $2$.

*(With periodic boundaries you would add a third bond $H_3 = J(X_3X_1+Y_3Y_1+Z_3Z_1)$ and the
splitting/commutator analysis would involve all three terms.)*

In [ ]:
# YOUR CODE HERE

# (a) Verify Trotter error bound
# bound = t^2 / (2*r) * norm([A, B])
# compare with actual error err1[i]

# (b) Heisenberg Hamiltonian on 3 sites
# J = 1.0
# n_heis = 3
# Build H1 = J*(X1X2 + Y1Y2 + Z1Z2), H2 = J*(X2X3 + Y2Y3 + Z2Z3)
# Compute commutator norm


In [ ]:
#@title 🔒 Exercise 6.1 (locked)  {display-mode: "form"}
reveal_solution("6.1", key="PASTE-KEY-HERE")


### 6.3 Sparse Hamiltonian simulation

A Hamiltonian is **$d$-sparse** if each row has at most $d$ non-zero entries.
The simulation cost (via product formulas) is $O(d^2 \|H\|_{\max} t / \varepsilon)$ gates.

The key idea is that each non-zero entry $H_{jk}$ generates a simple 2-body interaction
between basis states $|j\rangle$ and $|k\rangle$, which can be simulated with $O(1)$ gates.


In [ ]:
# ── Simulate time evolution under a random sparse Hamiltonian ─────────────────
n_sp = 3;  N_sp = 2**n_sp

# Build a random 2-sparse Hermitian matrix
np.random.seed(42)
H_sparse_np = np.zeros((N_sp, N_sp), dtype=complex)
for j in range(N_sp):
    k = (j + 1) % N_sp   # couple j to j+1 (circular)
    v = np.random.randn() + 1j*np.random.randn()
    H_sparse_np[j, k] = v
    H_sparse_np[k, j] = np.conj(v)
    H_sparse_np[j, j] = np.random.randn()   # diagonal

H_sp = qt.Qobj(H_sparse_np, dims=[[2]*n_sp, [2]*n_sp])
print(f"‖H_sparse‖ = {H_sp.norm():.3f},  max |H_ij| = {np.max(np.abs(H_sparse_np)):.3f}")
print(f"Sparsity: {np.sum(np.abs(H_sparse_np) > 1e-10)} non-zeros (d=2 sparse per row)")

# Split as sum of 1-sparse terms (each term has at most 1 non-zero off-diagonal)
# Each pair (j,k) contributes a 2x2 interaction
t_sp = 1.5
U_exact_sp = (-1j * H_sp * t_sp).expm()

# Product formula: alternate diag and off-diag terms
# For simplicity use full Trotter on each non-zero pair
pairs = [(j, (j+1)%N_sp) for j in range(N_sp)]
r_list = [1, 2, 4, 8, 16, 32]
errs_sp = []
for r_sp in r_list:
    dt_sp = t_sp / r_sp
    U_step = qt.tensor([qt.qeye(2)]*n_sp)
    for j, k in pairs:
        # 2x2 interaction block for (j,k): build embedded 2x2 evolution
        h_jk = np.zeros((N_sp, N_sp), dtype=complex)
        h_jk[j,k] = H_sparse_np[j,k]; h_jk[k,j] = H_sparse_np[k,j]
        h_jk[j,j] = H_sparse_np[j,j]               # full diagonal (each j appears in exactly one pair)
        H_term = qt.Qobj(h_jk, dims=[[2]*n_sp,[2]*n_sp])
        U_step = (-1j * H_term * dt_sp).expm() * U_step
    U_trot = U_step
    for _ in range(r_sp-1):
        U_trot = U_step * U_trot
    errs_sp.append((U_exact_sp - U_trot).norm())

fig, ax = plt.subplots(figsize=(6, 3))
ax.loglog(r_list, errs_sp, 'o-')
ax.loglog(r_list, [errs_sp[0]/r for r in r_list], 'k--', alpha=0.4, label='∝ 1/r')
ax.set_xlabel('Trotter steps r'); ax.set_ylabel('error')
ax.set_title('Sparse Hamiltonian Trotter simulation'); ax.legend()
plt.tight_layout(); plt.show()


---
## Summary

| Topic | Key result |
|-------|-----------|
| Stone's theorem | Every $U_t = e^{-iHt}$ for unique Hermitian $H$; $G = e^{-2i\theta\sigma_y}$ in Grover subspace |
| 1st-order Trotter | $(e^{-iAt/r}e^{-iBt/r})^r \to e^{-i(A+B)t}$; error $O(\|[A,B]\|\,t^2/r)$ |
| Strang splitting | $(e^{-iAt/2r}e^{-iBt/r}e^{-iAt/2r})^r$; error $O(t^3/r^2)$ — halves the exponent |
| Sparse Hamiltonian | $d$-sparse $H$: $\tilde{O}(d^2\|H\|_{\max}t/\varepsilon)$ gates via sparse oracle + Trotter |

**Next:** Notebook 7 combines QFT, QPE, and Hamiltonian simulation in the HHL algorithm (§7).